In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import os

# ============================================================
# USER SETTINGS
# ============================================================
# ---- Paths - EDIT THESE ----
RESULTS_DIR  = "/path/to/results"
DENOISED_DIR = os.path.join(RESULTS_DIR, "01_denoised")      # denoise_pipe.sh + runmydtd.m output
REG_DIR      = os.path.join(RESULTS_DIR, "04_registration")  # registration_pipe.sh output
SAVE_DIR     = os.path.join(RESULTS_DIR, "06_figures", "representative")

CONTROL_SUB    = 'sub-30'
CONCUSSION_SUB = 'sub-49'

DWI_SLICE_IDX = {
    'sub-30': 19,
    'sub-49': 21,
}
PARAM_SLICE_IDX = 70

DWI_LABELS = {
    'B0':  'b=0 LTE',
    'LTE': 'b=1.5 LTE',
    'STE': 'b=1.5 STE',
}

PARAM_DISPLAY_LABELS = {
    'ufa':   r'$\mathrm{\mu}$FA',
    'op':    'OP',
    'vison': r'$\hat{V}(D_{\mathrm{iso}})$',
    'FA':    'FA',
    'MD':    'MD',
}

PARAM_CMAPS = {
    'ufa':   'RdBu_r',
    'op':    'RdBu_r',
    'vison': 'magma',
    'FA':    'RdBu_r',
    'MD':    'magma',
}

PARAM_RANGES = {
    'ufa':   (0, 1),
    'op':    (0, 1),
    'vison': (0, 1),
    'FA':    (0, 1),
    'MD':    (0, 3),
}

PARAM_UNITS = {
    'MD':    r'$\mu$m²/ms',
    'vison': '',
    'ufa':   '',
    'op':    '',
    'FA':    '',
}

PARAM_SCALE = {
    'MD':   1.0 * 1000,
}

DTD_PARAMS = ['ufa', 'op', 'vison']
DTI_PARAMS = ['FA', 'MD']

# parameter row layout for combined figure
TOP_PARAMS = ['ufa', 'FA', 'op', 'vison', 'MD']       # top row of each subject block
BOT_PARAMS = []   # bottom row of each subject block
# ============================================================


def load_volume_map(sub_id):
    vol_map_path = os.path.join(
        DENOISED_DIR, sub_id, 'dwi', f'{sub_id}_volume_mapping.txt'
    )
    rows = []
    with open(vol_map_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            rows.append({
                'start':    int(parts[0]),
                'end':      int(parts[1]),
                'acq_type': parts[2],
                'filename': parts[3],
            })
    return pd.DataFrame(rows)


def load_bvals(sub_id):
    bval_path = os.path.join(
        DENOISED_DIR, sub_id, 'dwi', f'{sub_id}_concat_dwi.bval'
    )
    return np.loadtxt(bval_path)


def get_volume_indices(vol_map, bvals, acq_type_filter, bval_threshold=50, high_b=True):
    indices = []
    for _, row in vol_map.iterrows():
        if acq_type_filter and row['acq_type'].lower() != acq_type_filter.lower():
            continue
        for vol_idx in range(row['start'], row['end'] + 1):
            if vol_idx < len(bvals):
                bval = bvals[vol_idx]
                if high_b and bval > bval_threshold:
                    indices.append(vol_idx)
                elif not high_b and bval < bval_threshold:
                    indices.append(vol_idx)
    return indices


def get_mean_volume(dwi_data, indices):
    vols = dwi_data[..., indices]
    return np.mean(vols, axis=-1) if vols.ndim == 4 else vols


def get_axial_slice(data_3d, slice_idx, affine=None, extra_flipud=False):
    sl = data_3d[:, :, slice_idx]
    if affine is not None:
        if affine[0, 0] < 0:
            sl = np.fliplr(sl)
        if affine[1, 1] < 0:
            sl = np.flipud(sl)
    sl = np.rot90(sl)
    sl = np.flipud(sl)
    if extra_flipud:
        sl = np.flipud(sl)
    return sl


def get_crop_bounds(sl, pad=5):
    mask = sl > 0
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any() or not cols.any():
        return 0, sl.shape[0], 0, sl.shape[1]
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    rmin = max(0, rmin - pad)
    rmax = min(sl.shape[0], rmax + pad)
    cmin = max(0, cmin - pad)
    cmax = min(sl.shape[1], cmax + pad)
    return rmin, rmax, cmin, cmax


def apply_crop(sl, bounds):
    rmin, rmax, cmin, cmax = bounds
    return sl[rmin:rmax, cmin:cmax]


def add_row_label(ax, label, fontsize=18):
    ax.text(
        -0.15, 0.5, label,
        transform=ax.transAxes,
        fontsize=fontsize, fontweight='bold',
        va='center', ha='right',
        rotation=90,
    )


def get_param_path(sub_id, param):
    is_dtd = param in DTD_PARAMS
    if is_dtd:
        return os.path.join(
            REG_DIR, sub_id, "dtd_CONspace",
            f"{sub_id}_dtd_{param}_CONspace.nii.gz"
        )
    else:
        return os.path.join(
            REG_DIR, sub_id, "dti_CONspace",
            f"{sub_id}_{param}_CONspace.nii.gz"
        )


def get_crop_bounds_from_ufa(sub_id, pad=5):
    ufa_path = get_param_path(sub_id, 'ufa')
    if not os.path.exists(ufa_path):
        print(f"  WARNING: ufa not found for {sub_id}, crop bounds unavailable")
        return None
    img    = nib.load(ufa_path)
    data   = img.get_fdata(dtype=np.float32)
    sl_ufa = get_axial_slice(data, PARAM_SLICE_IDX, affine=img.affine)
    return get_crop_bounds(sl_ufa, pad=pad)


def plot_raw_dwi(sub_id, ax_row, axes, group_label):
    dwi_dir   = os.path.join(DENOISED_DIR, sub_id, 'dwi')
    orig_path = os.path.join(dwi_dir, f'{sub_id}_concat_dwi_original.nii.gz')
    den_path  = os.path.join(dwi_dir, f'{sub_id}_concat_dwi_denoised_mc.nii.gz')
    slice_idx = DWI_SLICE_IDX.get(sub_id, 19)

    for path in [orig_path, den_path]:
        if not os.path.exists(path):
            print(f"  Missing: {path}")
            return

    orig_img = nib.load(orig_path)
    den_img  = nib.load(den_path)
    orig     = orig_img.get_fdata(dtype=np.float32)
    den      = den_img.get_fdata(dtype=np.float32)
    affine   = orig_img.affine

    vol_map = load_volume_map(sub_id)
    bvals   = load_bvals(sub_id)

    b0_idx  = get_volume_indices(vol_map, bvals, acq_type_filter='linear',    bval_threshold=50,   high_b=False)
    lte_idx = get_volume_indices(vol_map, bvals, acq_type_filter='linear',    bval_threshold=1400, high_b=True)
    ste_idx = get_volume_indices(vol_map, bvals, acq_type_filter='spherical', bval_threshold=1400, high_b=True)

    print(f"  {sub_id}: b0={len(b0_idx)}, LTE={len(lte_idx)}, STE={len(ste_idx)}")

    volumes   = {'B0': b0_idx, 'LTE': lte_idx, 'STE': ste_idx}
    col       = 0
    first_col = True

    for vol_key, idx in volumes.items():
        vol_label = DWI_LABELS.get(vol_key, vol_key)

        if not idx:
            print(f"  No {vol_key} volumes found for {sub_id}")
            for c in [col, col + 1]:
                axes[ax_row][c].axis('off')
                axes[ax_row][c].set_title(f'{vol_label}\n(missing)', fontsize=13)
            col += 2
            first_col = False
            continue

        orig_vol = get_axial_slice(get_mean_volume(orig, idx), slice_idx, affine=affine, extra_flipud=True)
        den_vol  = get_axial_slice(get_mean_volume(den,  idx), slice_idx, affine=affine, extra_flipud=True)
        bounds   = get_crop_bounds(orig_vol, pad=5)
        orig_vol = apply_crop(orig_vol, bounds)
        den_vol  = apply_crop(den_vol,  bounds)

        vmax = np.percentile(orig_vol[orig_vol > 0], 99)

        axes[ax_row][col].imshow(orig_vol, cmap='gray', vmin=0, vmax=vmax, origin='lower')
        axes[ax_row][col].set_title(f'{vol_label} Raw', fontsize=14, fontweight='bold')
        axes[ax_row][col].axis('off')

        axes[ax_row][col + 1].imshow(den_vol, cmap='gray', vmin=0, vmax=vmax, origin='lower')
        axes[ax_row][col + 1].set_title(f'{vol_label} Denoised', fontsize=14, fontweight='bold')
        axes[ax_row][col + 1].axis('off')

        if first_col:
            add_row_label(axes[ax_row][col], group_label, fontsize=18)
            first_col = False

        col += 2


def plot_param_maps_combined(save_dir=None):
    """
    Layout: 2 rows x 5 cols
    Row 0: Control    — uFA | FA | OP | V(Diso) | MD
    Row 1: Concussion — uFA | FA | OP | V(Diso) | MD
    Row label ('Control'/'Concussion') on left of each row.
    """
    subjects = [
        (CONTROL_SUB,    'Control'),
        (CONCUSSION_SUB, 'Concussion'),
    ]

    n_cols = len(TOP_PARAMS)   # 3
    n_rows = 2                 # 2 subjects × 2 param rows each

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 3.8, n_rows * 3.8),
        gridspec_kw={'hspace': 0.2, 'wspace': 0.2} 
    )

    # precompute crop bounds per subject from ufa
    crop_bounds = {
        sub_id: get_crop_bounds_from_ufa(sub_id, pad=5)
        for sub_id, _ in subjects
    }

    # (figure row, subject, param list, show col titles, show row label, label)
    row_layout = [
        (0, CONTROL_SUB,    TOP_PARAMS, True,  True, 'Control'),
        (1, CONCUSSION_SUB, TOP_PARAMS, False, True, 'Concussion')
    ]

    for fig_row, sub_id, params, show_titles, show_row_label, group_label in row_layout:
        bounds = crop_bounds.get(sub_id)

        # turn off any columns beyond what this row's param list uses
        for empty_col in range(len(params), n_cols):
            axes[fig_row][empty_col].axis('off')

        for col_idx, param in enumerate(params):
            ax   = axes[fig_row][col_idx]
            path = get_param_path(sub_id, param)

            if not os.path.exists(path):
                print(f"  Missing: {path}")
                ax.axis('off')
                ax.set_title(f'{param}\n(missing)', fontsize=12)
                continue

            img    = nib.load(path)
            data   = img.get_fdata(dtype=np.float32)
            affine = img.affine

            sl    = get_axial_slice(data, PARAM_SLICE_IDX, affine=affine)
            scale = PARAM_SCALE.get(param, 1.0)
            sl    = sl * scale

            if bounds is not None:
                sl = apply_crop(sl, bounds)

            cmap       = PARAM_CMAPS.get(param, 'viridis')
            vmin, vmax = PARAM_RANGES.get(param, (None, None))

            valid = sl[np.isfinite(sl) & (sl > 0)]
            if len(valid) == 0:
                print(f"  WARNING: no valid voxels for {param} {sub_id}")
                ax.axis('off')
                continue

            if vmin is None:
                vmin = np.nanpercentile(valid, 1)
            if vmax is None:
                vmax = np.nanpercentile(valid, 99)

            im = ax.imshow(sl, cmap=cmap, vmin=vmin, vmax=vmax, origin='lower')
            ax.axis('off')
            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=0.8)
            cbar.ax.tick_params(labelsize=16)
            cbar.locator = plt.MaxNLocator(nbins=3)
            cbar.update_ticks()

            units = PARAM_UNITS.get(param, '')
            if units:
                cbar.ax.set_title(units, fontsize=16, pad=15, loc='left')


            # column titles on top row only
            if show_titles:
                display_label = PARAM_DISPLAY_LABELS.get(param, param)
                ax.set_title(display_label, fontsize=22, pad=8)

            # row label on leftmost column of first param row per subject
            if col_idx == 0 and show_row_label:
                add_row_label(ax, group_label, fontsize=22)

    plt.tight_layout(rect=[0.06, 0, 1, 1])

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fname = os.path.join(save_dir, 'param_maps_combined.png')
        plt.savefig(fname, dpi=600, bbox_inches='tight')
        print(f"  ✓ saved: {fname}")
    else:
        plt.show()
    plt.close()


def plot_raw_dwi_panel(save_dir=None):
    n_cols = 6
    n_rows = 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.2, n_rows * 3.5))
    plot_raw_dwi(CONTROL_SUB,    0, axes, 'Control')
    plot_raw_dwi(CONCUSSION_SUB, 1, axes, 'Concussion')
    plt.tight_layout(rect=[0.06, 0, 1, 1])
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fname = os.path.join(save_dir, 'raw_vs_denoised.png')
        plt.savefig(fname, dpi=600, bbox_inches='tight')
        print(f"  ✓ saved: {fname}")
    else:
        plt.show()
    plt.close()


# ============================================================
# RUN
# ============================================================
plot_raw_dwi_panel(save_dir=SAVE_DIR)
plot_param_maps_combined(save_dir=SAVE_DIR)